#### Imports and Load Data

Study area: the City of Los Angeles (LAPD jurisdiction), 2020 census tracts with >= 50 % of their area inside
the city. County tracts outside the city receive no LAPD reports and are therefore excluded.

In [ ]:
import os
import sys
import libpysal
import mapclassify
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.cm as cm
import matplotlib.pyplot as plt
from IPython.display import HTML
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
import matplotlib.animation as animation

sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..")))
import config
config.apply_plot_style()

In [ ]:
tracts = config.load_tracts(city_only=True)
geoids = tracts["GEOID"].tolist()
df_sp = config.load_theft_data(coords_required=True)
gdf = gpd.GeoDataFrame(df_sp, geometry=gpd.points_from_xy(df_sp["lon_final"], df_sp["lat_final"]),
                       crs=config.LATLON_CRS).to_crs(config.SPATIAL_CRS)
in_tract = gdf["GEOID"].isin(geoids)
print(f"City tracts                    : {len(tracts):,}")
print(f"Theft points (inside the city) : {len(gdf):,}")
print(f"  of which in an analysis tract: {in_tract.sum():,}  (rest fall in tracts < 50 % inside the city)")

# Tract x year count matrix (zeros included)
counts = (gdf[in_tract].groupby(["GEOID", "year"]).size().unstack(fill_value=0)
          .reindex(index=geoids, columns=config.STUDY_YEARS, fill_value=0))
veh = config.annual_denominator("vehicles", geoids)
pop = config.annual_denominator("population", geoids)

MIN_VEH, MIN_POP = 100, 500
tracts_plot = tracts.copy()
tracts_plot["total_count"] = counts.sum(axis=1).values
tracts_plot["avg_annual"] = tracts_plot["total_count"] / len(config.STUDY_YEARS)
tracts_plot["mean_vehicles"] = veh.mean(axis=1).values
tracts_plot["mean_population"] = pop.mean(axis=1).values
# Pooled rate = all thefts / all vehicle-years (annual denominators interpolated between vintages); years with zero thefts count.
tracts_plot["rate_per_1000"] = (counts.sum(axis=1) / veh.sum(axis=1) * 1000).values
tracts_plot["rate_per_100k"] = (counts.sum(axis=1) / pop.sum(axis=1) * 100_000).values
# Rates need a denominator in every year and are not defined for special land-use tracts (98xx: airport, port, parks)
special = config.is_special_use(tracts_plot["GEOID"])
tracts_plot.loc[(tracts_plot["mean_vehicles"].fillna(0) < MIN_VEH) | veh.isna().any(axis=1).values | special, "rate_per_1000"] = np.nan
tracts_plot.loc[(tracts_plot["mean_population"].fillna(0) < MIN_POP) | pop.isna().any(axis=1).values | special, "rate_per_100k"] = np.nan
print(f"Special land-use tracts excluded from rates: {special.sum()}")
print(tracts_plot[["total_count", "avg_annual", "rate_per_1000", "rate_per_100k"]].describe().round(1).to_string())

#### Choropleth helper

In [ ]:
def choropleth(col, title, legend_title, fname, low_label=None, k=5, fmt="{:.1f}"):
    tw = tracts_plot.copy()
    has = tw[col].notna() & (tw[col] > 0)
    vals = tw.loc[has, col]
    clf = mapclassify.NaturalBreaks(vals.clip(upper=np.percentile(vals, 98)), k=k)
    tw["_class"] = np.nan
    tw.loc[has, "_class"] = clf.yb.astype(float)
    colors = [plt.get_cmap("viridis")(p) for p in np.linspace(0.15, 0.92, k)]
    edges = [vals.min()] + list(clf.bins[:-1]) + [vals.max()]
    labels = [f"{fmt.format(edges[i])} - {fmt.format(edges[i+1])}  (n={(clf.yb == i).sum()})" for i in range(k)]
    fig, ax = plt.subplots(figsize=config.FIGURE_SIZE_SQUARE)
    tw[~has].plot(ax=ax, color="#aaaaaa", edgecolor="#888", linewidth=0.3, zorder=2)
    tw[has].plot(ax=ax, column="_class", cmap=mcolors.ListedColormap(colors),
                 norm=mcolors.BoundaryNorm(range(k + 1), k), linewidth=0.15, edgecolor="#bbb", zorder=3)
    config.add_basemap(ax)
    ax.set_axis_off()
    ax.set_title(title, pad=14)
    handles = [mpatches.Patch(facecolor=colors[i], edgecolor="#888", label=labels[i]) for i in range(k)]
    if (~has).sum():
        handles.append(mpatches.Patch(facecolor="#aaaaaa", edgecolor="#888", label=f"{low_label or 'no data'} (n={(~has).sum()})"))
    ax.legend(handles=handles, title=legend_title, loc="lower left", framealpha=0.9, edgecolor="#ccc")
    plt.tight_layout()
    plt.savefig(os.path.join(config.FIGURES_DIR, fname), dpi=config.FIGURE_DPI, bbox_inches="tight")
    plt.show()

#### Average annual thefts per tract

In [ ]:
choropleth("avg_annual", "Average Annual Vehicle Thefts per Census Tract (City of LA, 2000-2022)",
           "Thefts per year", "fig9_absolute_thefts.png", low_label="no thefts")

#### Thefts per 1,000 household vehicles

Denominator: vehicles available to resident households (Census 2000 H046 / ACS B25046), interpolated between vintages and
re-allocated to 2020 tracts. This is exposure of residents' vehicles, not of all vehicles parked in a tract;
commercial and transit areas with few residents get high rates for that reason.

In [ ]:
choropleth("rate_per_1000", "Annual Thefts per 1,000 Household Vehicles (pooled 2000-2022)",
           "Thefts / 1,000 vehicles / yr", "fig10_thefts_per1000vehicles.png", low_label=f"special land use or < {MIN_VEH} vehicles")

In [ ]:
choropleth("rate_per_100k", "Annual Thefts per 100,000 Residents (pooled 2000-2022)",
           "Thefts / 100k residents / yr", "fig11_thefts_per100k.png", low_label=f"special land use or < {MIN_POP} residents", fmt="{:.0f}")

#### Fixed-bandwidth KDE on a fixed city grid

All surfaces share one grid (100 m cells, Gaussian sigma 500 m, masked to the city) and are normalised to sum
to 1, so each cell holds its share of the period's thefts and periods with different volumes are comparable.

In [ ]:
grid = config.city_grid()
X, Y = gdf.geometry.x.values, gdf.geometry.y.values
yr = gdf["year"].values
z_all = config.kde_surface(X, Y, grid)
print(f"Grid: {grid['mask'].shape[1]} x {grid['mask'].shape[0]} cells of {config.KDE_CELL_M} m, "
      f"{grid['mask'].sum():,} inside the city")


def show_surface(z, title, fname, cmap="viridis", norm=None, cbar_label="Share of thefts per cell (relative)"):
    fig, ax = plt.subplots(figsize=config.FIGURE_SIZE_SQUARE)
    ax.set_xlim(grid["extent"][:2]); ax.set_ylim(grid["extent"][2:])
    config.add_basemap(ax, alpha=0.6)
    im = ax.imshow(z, origin="lower", extent=grid["extent"], cmap=cmap, norm=norm, alpha=0.75, zorder=3,
                   interpolation="nearest")
    ax.set_axis_off()
    ax.set_title(title, pad=14)
    cbar = plt.colorbar(im, ax=ax, fraction=0.03, pad=0.02, shrink=0.75)
    cbar.set_label(cbar_label)
    plt.tight_layout()
    plt.savefig(os.path.join(config.FIGURES_DIR, fname), dpi=config.FIGURE_DPI, bbox_inches="tight")
    plt.show()


vmax = np.nanpercentile(z_all, 99.5)
show_surface(np.ma.masked_where(~(z_all > np.nanpercentile(z_all, 15)), z_all),
             f"Vehicle Theft Density 2000-2022 (KDE, sigma {config.KDE_BANDWIDTH_M} m)", "fig12_kde_full.png",
             norm=mcolors.Normalize(0, vmax))

#### KDE difference maps

Main comparison uses two periods with the **same coordinate regime** (stored LAPD coordinates):
2002-2009 vs 2010-2017. A comparison involving 2018-2022 (geocoded locations) is shown separately and labelled,
because part of any difference there reflects the change of location source.

In [ ]:
def kde_diff(years_a, years_b):
    za = config.kde_surface(X[np.isin(yr, years_a)], Y[np.isin(yr, years_a)], grid)
    zb = config.kde_surface(X[np.isin(yr, years_b)], Y[np.isin(yr, years_b)], grid)
    return zb - za


EARLY, LATE, GEO = list(range(2002, 2010)), list(range(2010, 2018)), list(range(2018, 2023))
diffs = {
    "fig13_kde_diff.png": ("KDE Difference: 2010-2017 minus 2002-2009 (both stored coordinates)", kde_diff(EARLY, LATE)),
    "fig13b_kde_diff_geocoded_regime.png": ("KDE Difference: 2018-2022 minus 2010-2017 (later period geocoded)",
                                            kde_diff(LATE, GEO)),
    # Baseline: redistribution between two halves of one stable regime, to calibrate what "no change" looks like
    "fig13c_kde_diff_reference.png": ("KDE Difference: 2014-2017 minus 2010-2013 (reference)",
                                      kde_diff(list(range(2010, 2014)), list(range(2014, 2018)))),
}
# One colour scale for all three maps, so they can be compared directly
vabs = max(np.nanpercentile(np.abs(d), 99) for _, d in diffs.values())
for fname, (title, d) in diffs.items():
    show_surface(np.ma.masked_where(~(np.abs(d) >= vabs * 0.05), d), title, fname, cmap="RdBu_r",
                 norm=mcolors.TwoSlopeNorm(vmin=-vabs, vcenter=0, vmax=vabs),
                 cbar_label="Change in share of thefts per cell")
    # Share of thefts moving: half the L1 distance between the two normalised surfaces
    print(f"{title}: share of density redistributed = {np.nansum(np.abs(d)) / 2 * 100:.1f}%")

#### KDE animation (fixed grid)

In [ ]:
yearly = {y: config.kde_surface(X[yr == y], Y[yr == y], grid) for y in config.STUDY_YEARS if (yr == y).sum() >= 100}
vmax_g = np.nanpercentile(np.stack(list(yearly.values())), 99.5)
fig_a, ax_a = plt.subplots(figsize=config.FIGURE_SIZE_SQUARE)
ax_a.set_xlim(grid["extent"][:2]); ax_a.set_ylim(grid["extent"][2:])
config.add_basemap(ax_a, alpha=0.6)
ax_a.set_axis_off()
first = min(yearly)
im_a = ax_a.imshow(yearly[first], origin="lower", extent=grid["extent"], cmap="viridis", vmin=0, vmax=vmax_g,
                   alpha=0.7, zorder=3, interpolation="nearest")
title_a = ax_a.set_title("", pad=10)
n_year = pd.Series(yr).value_counts()


def update_frame(y):
    im_a.set_data(yearly[y])
    regime = "geocoded locations" if y in config.GEOCODED_REGIME_YEARS else "stored LAPD locations"
    title_a.set_text(f"Vehicle Theft Density {y} ({n_year[y]:,} thefts, {regime})")
    return [im_a, title_a]


ani = animation.FuncAnimation(fig_a, update_frame, frames=sorted(yearly), interval=700, blit=False)
ani.save(os.path.join(config.FIGURES_DIR, "fig12b_kde_animation_2000_2022.gif"), writer="pillow", fps=1.5, dpi=100)
plt.close(fig_a)

#### Gi* hotspot analysis

Robustness grid over the two choices that actually change the statistic:
spatial weights (queen vs rook) and the variable (23-year theft count vs pooled rate per 1,000 household
vehicles).
Tracts without neighbours (islands) are reported and kept with only themselves in their neighbourhood.

In [ ]:
w_queen = libpysal.weights.Queen.from_dataframe(tracts_plot, use_index=False)
w_rook = libpysal.weights.Rook.from_dataframe(tracts_plot, use_index=False)
print(f"Queen: mean neighbours {w_queen.mean_neighbors:.2f}, islands {len(w_queen.islands)}")
print(f"Rook : mean neighbours {w_rook.mean_neighbors:.2f}, islands {len(w_rook.islands)}")


def run_gi(var, w, perms, subset=None):
    """Gi* on tracts_plot[var]; tracts outside `subset` (e.g. no valid rate) are left out of the analysis."""
    idx = np.arange(len(tracts_plot)) if subset is None else np.flatnonzero(subset)
    ww = w if subset is None else libpysal.weights.w_subset(w, idx)
    zs, ps = config.gi_star(tracts_plot[var].values[idx], ww, permutations=perms)
    z = np.full(len(tracts_plot), np.nan); p = np.full(len(tracts_plot), np.nan)
    z[idx], p[idx] = zs, ps
    return z, p


ROBUST_PERMS = 9999
rate_ok = tracts_plot["rate_per_1000"].notna().values
configs = {"queen_count": ("total_count", w_queen, None), "rook_count": ("total_count", w_rook, None),
           "queen_rate": ("rate_per_1000", w_queen, rate_ok), "rook_rate": ("rate_per_1000", w_rook, rate_ok)}
robust = {}
for name, (var, w, sub) in configs.items():
    z, p = run_gi(var, w, ROBUST_PERMS, sub)
    tiers = pd.Series(config.gi_tiers(z, p))
    tiers[np.isnan(z)] = "excluded"
    tracts_plot[f"gi_tier_{name}"] = tiers.values
    robust[name] = tiers.value_counts()
order = config.GI_TIER_ORDER + ["excluded"]
table3 = pd.DataFrame(robust).reindex(order).fillna(0).astype(int)
# Agreement of the binary HH (p<=0.05) classification with the canonical configuration
hh = {k: tracts_plot[f"gi_tier_{k}"].str.startswith("HH") for k in configs}
table3.loc["HH agreement with rook_count (%)"] = [round((hh[k] == hh["rook_count"]).mean() * 100, 1) for k in configs]
table3.to_csv(os.path.join(config.TABLES_DIR, "table3_gi_robustness.csv"))
print(f"Table 3 - Gi* tiers across weight/variable choices ({ROBUST_PERMS:,} permutations; smallest two-sided p = 0.0002)")
print(table3.to_string())

#### Canonical Gi* (rook, 23-year count, 9,999 permutations) with FDR control

Pseudo p-values are adjusted with Benjamini-Hochberg across all tracts. Downstream notebooks define a
**hotspot** as an HH tract with FDR q <= 0.05; the uncorrected HH p <= 0.05 set is saved for sensitivity checks.

In [ ]:
z, p = run_gi("total_count", w_rook, 9999)
tracts_plot["gi_z"], tracts_plot["gi_p"] = z, p
tracts_plot["gi_q"] = config.bh_fdr(p)
tracts_plot["gi_tier"] = config.gi_tiers(z, p)
tracts_plot["is_hotspot"] = (z > 0) & (tracts_plot["gi_q"] <= 0.05)
tracts_plot["is_coldspot"] = (z < 0) & (tracts_plot["gi_q"] <= 0.05)
tracts_plot["is_hotspot_p05"] = (z > 0) & (p <= 0.05 + 1e-9)   # uncorrected, for sensitivity checks

table4 = pd.DataFrame({"n_tracts": tracts_plot["gi_tier"].value_counts().reindex(config.GI_TIER_ORDER, fill_value=0)})
table4.loc["HH with FDR q<=0.05 (hotspot)"] = tracts_plot["is_hotspot"].sum()
table4.loc["LL with FDR q<=0.05 (coldspot)"] = tracts_plot["is_coldspot"].sum()
table4.to_csv(os.path.join(config.TABLES_DIR, "table4_gi_tiers_9999.csv"))
print(table4.to_string())
print(f"Smallest attainable two-sided p = 2/10000; tracts at the minimum: {(p <= 2e-4 + 1e-9).sum()}")

In [ ]:
fig, ax = plt.subplots(figsize=config.FIGURE_SIZE_SQUARE)
for tier in config.GI_TIER_ORDER:
    sub = tracts_plot[tracts_plot["gi_tier"] == tier]
    if len(sub):
        sub.plot(ax=ax, color=config.GI_TIER_COLORS[tier], edgecolor="#aaa", linewidth=0.15, zorder=3)
tracts_plot[tracts_plot["is_hotspot"]].boundary.plot(ax=ax, color="black", linewidth=0.5, zorder=4)
config.add_basemap(ax)
ax.set_axis_off()
ax.set_title("Gi* Hotspot / Coldspot Map, City of LA 2000-2022\n(rook contiguity, 9,999 permutations)", pad=14)
handles = [mpatches.Patch(facecolor=config.GI_TIER_COLORS[t], edgecolor="#888", label=f"{t}  (n={(tracts_plot['gi_tier'] == t).sum()})")
           for t in config.GI_TIER_ORDER if (tracts_plot["gi_tier"] == t).sum()]
handles.append(mpatches.Patch(facecolor="none", edgecolor="black", label=f"HH, FDR q<=0.05 (n={tracts_plot['is_hotspot'].sum()})"))
ax.legend(handles=handles, title="Gi* tier (pseudo p)", loc="lower left", framealpha=0.95, edgecolor="#ccc")
plt.tight_layout()
plt.savefig(os.path.join(config.FIGURES_DIR, "fig14_gi_star_map.png"), dpi=config.FIGURE_DPI, bbox_inches="tight")
plt.show()

#### Extreme hotspot tracts: are they driven by stacked points?

Many reports share identical coordinates (block centroids, stations, impound lots, geocoder fallbacks).
For each extreme tract, report the share of its thefts at the single most frequent coordinate.

In [ ]:
pts_in = gdf[in_tract].copy()
pts_in["xy"] = pts_in["lat_final"].round(5).astype(str) + "," + pts_in["lon_final"].round(5).astype(str)
top_loc = pts_in.groupby("GEOID")["xy"].agg(lambda s: s.value_counts().iloc[0] / len(s) * 100).rename("top_location_share_pct")
geocoded_share = pts_in.groupby("GEOID")["coord_source"].agg(lambda s: (s != "stored").mean() * 100).rename("geocoded_share_pct")
extreme = (tracts_plot[tracts_plot["gi_tier"] == "HH p<=0.001"]
           [["GEOID", "gi_tier", "gi_z", "gi_q", "avg_annual", "rate_per_1000", "mean_vehicles"]]
           .merge(top_loc, on="GEOID", how="left").merge(geocoded_share, on="GEOID", how="left")
           .sort_values("gi_z", ascending=False))
extreme.to_csv(os.path.join(config.TABLES_DIR, "table15_extreme_hotspots.csv"), index=False)
print(extreme.round(3).to_string(index=False))
print(f"City-wide median top-location share per tract: {top_loc.median():.1f}%")

In [ ]:
hh_ext = tracts_plot[tracts_plot["gi_tier"] == "HH p<=0.001"]
if len(hh_ext):
    fig, ax = plt.subplots(figsize=(10, 8))
    for tier in ["HH p<=0.01", "HH p<=0.05"]:
        tracts_plot[tracts_plot["gi_tier"] == tier].plot(ax=ax, color=config.GI_TIER_COLORS[tier], alpha=0.55, zorder=3)
    hh_ext.plot(ax=ax, color=config.GI_TIER_COLORS["HH p<=0.001"], edgecolor="black", linewidth=1.5, alpha=0.85, zorder=5)
    for _, r in hh_ext.iterrows():
        c = r.geometry.representative_point()
        ax.annotate(f"{r['avg_annual']:.0f}/yr", (c.x, c.y), ha="center", va="center", color="white", fontsize=10,
                    bbox=dict(boxstyle="round,pad=0.25", facecolor="black", alpha=0.35, edgecolor="none"), zorder=6)
    b = hh_ext.total_bounds
    pad = 4000 * config.FT_PER_M
    ax.set_xlim(b[0] - pad, b[2] + pad); ax.set_ylim(b[1] - pad, b[3] + pad)
    config.add_basemap(ax, zoom=14, alpha=0.8)
    ax.set_axis_off()
    fig.suptitle("Gi* Hotspot Tracts with p <= 0.001 (two-sided, 9,999 permutations)", y=1.0)
    plt.tight_layout()
    plt.savefig(os.path.join(config.FIGURES_DIR, "fig15_gi_star_extreme_hh_zoomed.png"), dpi=config.FIGURE_DPI, bbox_inches="tight")
    plt.show()

#### Sensitivity: do hotspots depend on the location source?

Re-run the canonical Gi* (a) using only years with stored LAPD coordinates (excluding 2001 and 2018-2022) and
(b) without Nominatim-geocoded points, and compare the FDR hotspot classification with the main result.

In [ ]:
from sklearn.metrics import cohen_kappa_score
stored_years = [y for y in config.STUDY_YEARS if y not in config.GEOCODED_REGIME_YEARS]
tracts_plot["count_stored_years"] = counts[stored_years].sum(axis=1).values
no_nom = gdf[in_tract & (gdf["coord_source"] != "nominatim")]
tracts_plot["count_no_nominatim"] = no_nom.groupby("GEOID").size().reindex(geoids, fill_value=0).values
hot_a = tracts_plot["is_hotspot"].values
rows_s = []
for scenario, col in [("stored-coordinate years only", "count_stored_years"), ("without Nominatim points", "count_no_nominatim")]:
    z_s, p_s = run_gi(col, w_rook, 9999)
    hot_s = (z_s > 0) & (config.bh_fdr(p_s) <= 0.05)
    rows_s.append({"scenario": scenario, "hotspots_main": int(hot_a.sum()), "hotspots_scenario": int(hot_s.sum()),
                   "both": int((hot_a & hot_s).sum()), "jaccard": (hot_a & hot_s).sum() / max((hot_a | hot_s).sum(), 1),
                   "cohen_kappa": cohen_kappa_score(hot_a, hot_s),
                   "spearman_z": pd.Series(z).corr(pd.Series(z_s), method="spearman")})
sens = pd.DataFrame(rows_s)
sens.to_csv(os.path.join(config.TABLES_DIR, "table19_gi_location_sensitivity.csv"), index=False)
print(sens.round(3).to_string(index=False))

#### Concentration of theft across places (and sensitivity to the spatial unit)

Share of all thefts falling in the top x % of spatial units, for 250 m grid cells and for census tracts.
Units are restricted to the city; cells/tracts with zero thefts count as units.

In [ ]:
cell_ft = 250 * config.FT_PER_M
g250 = config.city_grid(cell_m=250, pad_m=0)
h250, _, _ = np.histogram2d(Y, X, bins=[g250["yedges"], g250["xedges"]])
cell_counts = np.sort(h250[g250["mask"]])[::-1]
tract_counts = np.sort(tracts_plot["total_count"].values)[::-1]


def concentration(cnts):
    c = np.cumsum(cnts) / cnts.sum()
    share_units = np.arange(1, len(cnts) + 1) / len(cnts)
    return share_units, c


rows_c = []
fig, ax = plt.subplots(figsize=config.FIGURE_SIZE_REGULAR)
# Cells without thefts include mountains, parks and the airfield, so the curve is also shown for cells with >= 1 theft
for lab, cnts, col in [("250 m grid cells", cell_counts, "#b2182b"),
                       ("250 m grid cells with >= 1 theft", cell_counts[cell_counts > 0], "#ef8a62"),
                       ("Census tracts", tract_counts, "#2166ac")]:
    su, c = concentration(cnts)
    ax.plot(su * 100, c * 100, color=col, lw=2.2, label=f"{lab} (n={len(cnts):,})")
    for pct in [1, 5, 10, 25, 50]:
        rows_c.append({"unit": lab, "top_pct_of_units": pct, "share_of_thefts_pct": c[int(np.ceil(len(cnts) * pct / 100)) - 1] * 100})
    rows_c.append({"unit": lab, "top_pct_of_units": "units holding 50% of thefts (%)",
                   "share_of_thefts_pct": (np.searchsorted(c, 0.5) + 1) / len(cnts) * 100})
ax.plot([0, 100], [0, 100], color="#888", ls="--", lw=1, label="Uniform")
ax.set_xlabel("Top x % of spatial units (ranked by thefts)")
ax.set_ylabel("Cumulative share of thefts (%)")
ax.set_title("Concentration of Vehicle Theft, City of LA 2000-2022", pad=12)
ax.spines[["top", "right"]].set_visible(False)
ax.legend(frameon=False)
plt.tight_layout()
plt.savefig(os.path.join(config.FIGURES_DIR, "fig38_concentration_curve.png"), dpi=config.FIGURE_DPI, bbox_inches="tight")
plt.show()
table20 = pd.DataFrame(rows_c)
table20.to_csv(os.path.join(config.TABLES_DIR, "table20_concentration.csv"), index=False)
print(table20.round(1).to_string(index=False))

In [ ]:
# Save tract-level results for notebooks 04-06
out_cols = ["GEOID", "NAME", "ALAND", "AWATER", "total_count", "avg_annual", "mean_vehicles", "mean_population",
            "rate_per_1000", "rate_per_100k", "gi_z", "gi_p", "gi_q", "gi_tier", "is_hotspot", "is_coldspot", "is_hotspot_p05",
            "geometry"] + [c for c in tracts_plot.columns if c.startswith("gi_tier_")]
tracts_plot[out_cols].to_file(os.path.join(config.DATA_DIR, "tracts_plot_gi.gpkg"), driver="GPKG")
print(f"Saved tracts_plot_gi.gpkg: {len(tracts_plot)} tracts, {tracts_plot['is_hotspot'].sum()} hotspots (FDR)")